# Actividad 2 — Principio de Inversión de Dependencias (DIP)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Sistema de Notificación y Alarmas de Emergencia en Bodegas Frigoríficas

---

## 1. Ejemplo Incorrecto (Violando DIP)

En este diseño, la clase de alto nivel `SistemaAlarmaFrio` instancia directamente en su constructor a dos clases concretas de bajo nivel: `BuzzerHardware` y `ServicioSmsTwilio`.



In [ ]:
# Clases concretas de bajo nivel
class BuzzerHardware:
    def __init__(self, puerto_com: str, volumen_db: int = 100) -> None:
        self.puerto_com: str = puerto_com
        self.volumen_db: int = volumen_db

    def hacer_sonar(self, duracion_seg: int) -> str:
        return f"[SIRENA FÍSICA en {self.puerto_com}] Sonando a {self.volumen_db} dB por {duracion_seg}s."


class ServicioSmsTwilio:
    def __init__(self, account_sid: str, token: str) -> None:
        self.account_sid: str = account_sid
        self.token: str = token

    def mandar_sms(self, telefono: str, mensaje: str) -> str:
        return f"[SMS TWILIO a {telefono} con cuenta {self.account_sid[:4]}***] Texto: {mensaje}"


# Clase de alto nivel que viola DIP (se amarra a las clases de bajo nivel)
class SistemaAlarmaFrioRigido:
    def __init__(self, id_bodega: str) -> None:
        self.id_bodega: str = id_bodega
        # ACOPLAMIENTO DIRECTO: Crea las instancias concretas adentro
        self.sirena = BuzzerHardware(puerto_com="COM3", volumen_db=110)
        self.servicio_sms = ServicioSmsTwilio(account_sid="AC998877", token="secreto123")

    def notificar_emergencia(self, id_cuarto: str, temp_actual: float) -> None:
        print(f"[{self.id_bodega}] ALERTA CRÍTICA en {id_cuarto} a {temp_actual}°C")
        # Llama directamente a los métodos concretos
        msg1 = self.sirena.hacer_sonar(duracion_seg=5)
        msg2 = self.servicio_sms.mandar_sms("+573001234567", f"Fallo en {id_cuarto}: {temp_actual}°C")
        print(f" -> {msg1}")
        print(f" -> {msg2}")



Clase SistemaAlarmaFrioRigido creada.


In [2]:
# Probamos la versión rígida
print("--- Probando Sistema Acoplado (Sin DIP) ---")
alarma_rigida = SistemaAlarmaFrioRigido("BODEGA-CENTRAL-BOGOTA")
alarma_rigida.notificar_emergencia("CUARTO-VACUNAS-01", 11.5)


--- Probando Sistema Acoplado (Sin DIP) ---
[BODEGA-CENTRAL-BOGOTA] ALERTA CRÍTICA en CUARTO-VACUNAS-01 a 11.5°C
 -> [SIRENA FÍSICA en COM3] Sonando a 110 dB por 5s.
 -> [SMS TWILIO a +573001234567 con cuenta AC99***] Texto: Fallo en CUARTO-VACUNAS-01: 11.5°C


## 2. Ejemplo Correcto (Aplicando DIP)

Para aplicar la inversión de dependencias:
1. Creamos la abstracción `CanalAlerta(ABC)` con el método `enviar_mensaje(asunto, texto)`.
2. Hacemos que todas las formas de notificar (`NotificadorSirena`, `NotificadorSMS`, `NotificadorSlack`, y un `NotificadorMock` para pruebas) implementen esa abstracción.
3. La clase de alto nivel `SistemaAlarmaFrioModular` **recibe los canales inyectados por constructor**. No sabe ni le importa cómo funciona cada canal por dentro.


In [2]:
from abc import ABC, abstractmethod

# 1. Abstracción común (Contrato independiente)
class CanalAlerta(ABC):
    @abstractmethod
    def enviar_mensaje(self, asunto: str, cuerpo: str) -> str:
        pass

    @abstractmethod
    def esta_disponible(self) -> bool:
        pass


# 2. Implementaciones concretas de bajo nivel
class NotificadorSirena(CanalAlerta):
    def __init__(self, puerto_com: str, volumen_db: int = 110) -> None:
        self.puerto_com: str = puerto_com
        self.volumen_db: int = volumen_db

    def enviar_mensaje(self, asunto: str, cuerpo: str) -> str:
        return f"[SIRENA en {self.puerto_com}] Pitido de alarma a {self.volumen_db} dB -> {asunto}"

    def esta_disponible(self) -> bool:
        return self.puerto_com.startswith("COM")


class NotificadorSMS(CanalAlerta):
    def __init__(self, numero_destino: str, proveedor: str = "Twilio") -> None:
        self.numero_destino: str = numero_destino
        self.proveedor: str = proveedor

    def enviar_mensaje(self, asunto: str, cuerpo: str) -> str:
        return f"[SMS {self.proveedor} a {self.numero_destino}] {asunto}: {cuerpo}"

    def esta_disponible(self) -> bool:
        return len(self.numero_destino) >= 10


class NotificadorSlack(CanalAlerta):
    def __init__(self, webhook_url: str, canal: str = "#alertas-frio") -> None:
        self.webhook_url: str = webhook_url
        self.canal: str = canal

    def enviar_mensaje(self, asunto: str, cuerpo: str) -> str:
        return f"[SLACK en {self.canal}] Webhook: *{asunto}* -> {cuerpo}"

    def esta_disponible(self) -> bool:
        return self.webhook_url.startswith("https://")


# Canal simulado para pruebas unitarias sin gastar plata ni usar hardware
class NotificadorMockParaPruebas(CanalAlerta):
    def __init__(self, nombre_test: str) -> None:
        self.nombre_test: str = nombre_test
        self.mensajes_guardados: list = []

    def enviar_mensaje(self, asunto: str, cuerpo: str) -> str:
        self.mensajes_guardados.append({"asunto": asunto, "cuerpo": cuerpo})
        return f"[MOCK TEST: {self.nombre_test}] Mensaje capturado en memoria (Total: {len(self.mensajes_guardados)})"

    def esta_disponible(self) -> bool:
        return True


# 3. Módulo de alto nivel (Solo depende de la abstracción CanalAlerta)
class SistemaAlarmaFrioModular:
    def __init__(self, id_bodega: str, canales: list = None) -> None:
        self.id_bodega: str = id_bodega
        # Recibe la lista de canales inyectada
        self.canales: list = canales if canales is not None else []

    def agregar_canal(self, canal: CanalAlerta) -> None:
        self.canales.append(canal)

    def disparar_alarma(self, id_cuarto: str, temp_actual: float, temp_limite: float) -> list:
        asunto = f"EMERGENCIA TÉRMICA en {id_cuarto}"
        cuerpo = f"Temperatura subió a {temp_actual}°C (Límite permitido: {temp_limite}°C) en {self.id_bodega}."

        respuestas = []
        for canal in self.canales:
            if canal.esta_disponible():
                resp = canal.enviar_mensaje(asunto, cuerpo)
                respuestas.append(resp)
            else:
                respuestas.append(f"[Aviso] Canal {type(canal).__name__} no disponible.")
        return respuestas


In [3]:
# Demostración
print("=== [1] CASO DE PRODUCCIÓN REAL (Sirena + SMS + Slack) ===")
canales_reales = [
    NotificadorSirena("COM2", volumen_db=115),
    NotificadorSMS("+573109876543", proveedor="TwilioGateway"),
    NotificadorSlack("https://hooks.slack.com/services/T00/B00/X00", canal="#guardia-frio")
]

alarma_prod = SistemaAlarmaFrioModular("BODEGA-CENTRAL-BOGOTA", canales=canales_reales)
envios_prod = alarma_prod.disparar_alarma("CUARTO-VACUNAS-01", temp_actual=9.5, temp_limite=8.0)

for res in envios_prod:
    print(f" -> {res}")


print("\n=== [2] CASO DE PRUEBA UNITARIA (Sin hardware ni costo de SMS) ===")
canal_test = NotificadorMockParaPruebas(nombre_test="TEST-01-DESCONEXION")
alarma_test = SistemaAlarmaFrioModular("BODEGA-PRUEBAS", canales=[canal_test])

envios_test = alarma_test.disparar_alarma("CUARTO-PRUEBA-X", temp_actual=14.0, temp_limite=8.0)
for res in envios_test:
    print(f" -> {res}")

print(f"\nComprobación de testing: Mensajes capturados = {len(canal_test.mensajes_guardados)}")
print(f"Contenido del mensaje mock: {canal_test.mensajes_guardados[0]['asunto']}")


=== [1] CASO DE PRODUCCIÓN REAL (Sirena + SMS + Slack) ===
 -> [SIRENA en COM2] Pitido de alarma a 115 dB -> EMERGENCIA TÉRMICA en CUARTO-VACUNAS-01
 -> [SMS TwilioGateway a +573109876543] EMERGENCIA TÉRMICA en CUARTO-VACUNAS-01: Temperatura subió a 9.5°C (Límite permitido: 8.0°C) en BODEGA-CENTRAL-BOGOTA.
 -> [SLACK en #guardia-frio] Webhook: *EMERGENCIA TÉRMICA en CUARTO-VACUNAS-01* -> Temperatura subió a 9.5°C (Límite permitido: 8.0°C) en BODEGA-CENTRAL-BOGOTA.

=== [2] CASO DE PRUEBA UNITARIA (Sin hardware ni costo de SMS) ===
 -> [MOCK TEST: TEST-01-DESCONEXION] Mensaje capturado en memoria (Total: 1)

Comprobación de testing: Mensajes capturados = 1
Contenido del mensaje mock: EMERGENCIA TÉRMICA en CUARTO-PRUEBA-X
